In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv("../data/processed_data.csv")
print(df.shape)
df.head()

(10194, 31)


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Country/Region,City,State/Province,Postal Code,...,Customer Lat,Customer Lon,Factory,Factory Lat,Factory Lon,Distance (km),Base Days,Distance Days,Noise,Synthetic Lead Time
0,1,US-2021-103800-CHO-MIL-31000,2024-01-03,2026-06-30,Standard Class,103800,United States,Houston,Texas,77095,...,29.758938,-95.367697,Wicked Choccy's,32.076176,-81.088371,1385.223052,3,1.731529,0.248357,5.0
1,2,US-2021-112326-CHO-TRI-54000,2024-01-04,2026-07-01,Standard Class,112326,United States,Naperville,Illinois,60540,...,41.772870,-88.147928,Wicked Choccy's,32.076176,-81.088371,1246.456971,3,1.558071,-0.069132,5.0
2,3,US-2021-112326-CHO-NUT-13000,2024-01-04,2026-07-01,Standard Class,112326,United States,Naperville,Illinois,60540,...,41.772870,-88.147928,Lot's O' Nuts,32.881893,-111.768036,2300.484764,3,2.875606,0.323844,7.0
3,4,US-2021-112326-CHO-SCR-58000,2024-01-04,2026-07-01,Standard Class,112326,United States,Naperville,Illinois,60540,...,41.772870,-88.147928,Lot's O' Nuts,32.881893,-111.768036,2300.484764,3,2.875606,0.761515,7.0
4,5,US-2021-141817-CHO-TRI-54000,2024-01-05,2026-07-05,Standard Class,141817,United States,Philadelphia,Pennsylvania,19143,...,39.952724,-75.163526,Wicked Choccy's,32.076176,-81.088371,1024.603225,3,1.280754,-0.117077,5.0


In [3]:
features = ["Ship Mode", "Origin Factory", "Region", "Division", "Distance (km)"]
target = "Synthetic Lead Time"

X = df[features]
y = df[target]

X_encoded = pd.get_dummies(X, columns = ["Ship Mode", "Origin Factory", "Region", "Division"], drop_first = True)

print(X_encoded.shape)
X_encoded.head()

(10194, 13)


,Distance (km),Ship Mode_Same Day,Ship Mode_Second Class,Ship Mode_Standard Class,Origin Factory_Secret Factory,Origin Factory_Sugar Shack,Origin Factory_The Other Factory,Origin Factory_Wicked Choccy's,Region_Gulf,Region_Interior,Region_Pacific,Division_Other,Division_Sugar
0,1385.223052,False,False,True,False,False,False,True,False,True,False,False,False
1,1246.456971,False,False,True,False,False,False,True,False,True,False,False,False
2,2300.484764,False,False,True,False,False,False,False,False,True,False,False,False
3,2300.484764,False,False,True,False,False,False,False,False,True,False,False,False
4,1024.603225,False,False,True,False,False,False,True,False,False,False,False,False


In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded,
    y,
    test_size= 0.2,
    random_state= 42
)

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)

Train size: (8155, 13)
Test size: (2039, 13)


In [8]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

y_pred_lr =lr_model.predict(X_test)

rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))
mae_lr = mean_absolute_error(y_test, y_pred_lr)
r2_lr = r2_score(y_test, y_pred_lr)

print(f"Model successfully trained! \nRMSE: {rmse_lr:.3f}, MAE: {mae_lr:.3f}, R²: {r2_lr:.3f}")

Model successfully trained! 
RMSE: 0.574, MAE: 0.452, R²: 0.894


In [9]:
from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(n_estimators=100, random_state= 42)
rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)

rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
mae_rf = mean_absolute_error(y_test, y_pred_rf)
r2_rf = r2_score(y_test, y_pred_rf)

print(f"Model successfully trained! \nRMSE: {rmse_rf:.3f}, MAE: {mae_rf:.3f}, R²: {r2_rf:.3f}")


Model successfully trained! 
RMSE: 0.622, MAE: 0.482, R²: 0.875


In [10]:
from sklearn.ensemble import GradientBoostingRegressor

gb_model = GradientBoostingRegressor(n_estimators=100, random_state= 42)
gb_model.fit(X_train, y_train)

y_pred_gb = gb_model.predict(X_test)

rmse_gb = np.sqrt(mean_squared_error(y_test, y_pred_gb))
mae_gb = mean_absolute_error(y_test, y_pred_gb)
r2_gb = r2_score(y_test, y_pred_gb)

print(f"Model successfully trained! \nRMSE: {rmse_gb:.3f}, MAE: {mae_gb:.3f}, R²: {r2_gb:.3f}")

Model successfully trained! 
RMSE: 0.578, MAE: 0.458, R²: 0.892


In [11]:
results = pd.DataFrame({
    "Model": ["Linear Regression", "Random Forest","Gradient Boosting"],
    "RMSE": [rmse_lr, rmse_rf, rmse_gb],
    "MAE": [mae_lr, mae_rf, mae_gb],
    "R²": [r2_lr, r2_rf, r2_gb]
}).sort_values("R²", ascending= False)

results

,Model,RMSE,MAE,R²
0,Linear Regression,0.574144,0.451700,0.893730
2,Gradient Boosting,0.578000,0.458382,0.892298
1,Random Forest,0.621844,0.482158,0.875339


In [12]:
coefficients = pd.DataFrame({
    "Feature": X_encoded.columns,
    "Coefficient": lr_model.coef_
}).sort_values("Coefficient", ascending= False)

coefficients

,Feature,Coefficient
3,Ship Mode_Standard Class,2.007512
2,Ship Mode_Second Class,1.019246
5,Origin Factory_Sugar Shack,0.049927
12,Division_Sugar,0.007849
0,Distance (km),0.001244
9,Region_Interior,-0.012106
4,Origin Factory_Secret Factory,-0.019945
7,Origin Factory_Wicked Choccy's,-0.023117
8,Region_Gulf,-0.036224
10,Region_Pacific,-0.043854


In [ ]:
'''
import joblib
joblib.dump(lr_model, "../models/lead_time_model.pkl")
joblib.dump(list(X_encoded.columns), "../models/model_features.pkl")
print("Model saved!")
'''

Model saved!
